# 05 · Serie de tiempo: ¿se puede anticipar El Niño?
[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/andrezconz/investigacion-aumentada-ml/blob/main/notebooks/05_series_tiempo_el_nino.ipynb)

**El dato:** el Índice Oceánico El Niño (ONI) de la NOAA: anomalía mensual de la temperatura del mar en el Pacífico central desde 1950. Por encima de +0,5 °C durante cinco meses seguidos se declara un episodio de El Niño; por debajo de −0,5 °C, de La Niña.

**Familia:** supervisado. El pasado funciona como la respuesta conocida: el modelo aprende hasta diciembre de 2015 y debe pronosticar 2016 y 2017 sin verlos.

**Cómo funciona ARIMA:** predice el siguiente mes a partir de los meses anteriores y de los errores que cometió recientemente.

In [ ]:
import urllib.request, os
URL = "https://psl.noaa.gov/data/correlation/oni.data"
if not os.path.exists("oni.data"):
    urllib.request.urlretrieve(URL, "oni.data")

import pandas as pd, numpy as np
filas = []
for linea in open("oni.data"):
    p = linea.split()
    if len(p) == 13 and p[0].isdigit() and len(p[0]) == 4:
        anio = int(p[0])
        for mes, v in enumerate(p[1:], 1):
            v = float(v)
            if v > -90:
                filas.append((pd.Timestamp(anio, mes, 1), v))
oni = pd.Series(dict(filas)).sort_index()
print(len(oni), "meses ·", oni.index.min().date(), "a", oni.index.max().date())

In [ ]:
def episodios(serie, umbral, signo):
    cond = (serie >= umbral) if signo > 0 else (serie <= -umbral)
    bloques = (cond != cond.shift()).cumsum()
    largos = cond.groupby(bloques).agg(["first", "size"])
    return int(((largos["first"]) & (largos["size"] >= 5)).sum())

print("Episodios de El Niño:", episodios(oni, .5, 1), "· de La Niña:", episodios(oni, .5, -1))

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(15, 4))
plt.plot(oni, color="#020E60", lw=1)
plt.axhline(.5, color="#C0392B", ls="--", lw=1); plt.axhline(-.5, color="#2E5FAE", ls="--", lw=1)
plt.title("Índice Oceánico El Niño (ONI)"); plt.ylabel("°C"); plt.show()

## Entrenar hasta 2015 y pronosticar 24 meses
Comparamos contra una alternativa **ingenua**: repetir el último dato conocido. Si el modelo no le gana a eso, no sirve.

In [ ]:
import warnings, itertools
from statsmodels.tsa.arima.model import ARIMA
warnings.filterwarnings("ignore")

entrenamiento = oni[:"2015-12"]
prueba = oni["2016-01":"2017-12"]

mejor = None
for p, q in itertools.product(range(4), range(4)):          # escoger el orden por AIC
    try:
        m = ARIMA(entrenamiento.values, order=(p, 0, q)).fit()
        if mejor is None or m.aic < mejor[0]:
            mejor = (m.aic, (p, 0, q), m)
    except Exception:
        pass
print("Orden escogido:", mejor[1])

pron = mejor[2].forecast(len(prueba))
ingenuo = np.repeat(entrenamiento.iloc[-1], len(prueba))
rmse = lambda a, b: float(np.sqrt(np.mean((np.asarray(a) - np.asarray(b)) ** 2)))
print(f"Error del modelo (RMSE):   {rmse(prueba, pron):.2f} °C")
print(f"Error del ingenuo (RMSE):  {rmse(prueba, ingenuo):.2f} °C")

In [ ]:
plt.figure(figsize=(13, 5))
plt.plot(oni["2012":"2017-12"], color="#020E60", lw=2, label="Lo que pasó")
plt.plot(prueba.index, pron, color="#C0392B", ls="--", lw=2, label="Pronóstico ARIMA")
plt.plot(prueba.index, ingenuo, color="#D4A017", ls=":", lw=2, label="Repetir el último dato")
plt.axvline(pd.Timestamp("2015-12-01"), color="grey", ls=":"); plt.legend(); plt.ylabel("°C")
plt.title("El modelo solo vio hasta diciembre de 2015"); plt.show()

> **Tip:** compara siempre contra una alternativa simple. Y desconfía de un solo periodo de prueba: cambia las fechas de entrenamiento y mira si el modelo sigue ganando.

*Fuente: NOAA Physical Sciences Laboratory, Oceanic Niño Index. Elaboración propia. Los resultados pueden diferir levemente de los de la presentación, que se hicieron en R.*